# Customer Support Ticket Classifier
Task 29 — Capstone. Classifies support tickets into categories (Billing, Technical, Account, General) using TF-IDF + Logistic Regression, compared with an SVM.

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import joblib

pd.set_option('display.max_colwidth', 60)

## 1. Load dataset
Replace this cell with `pd.read_csv('your_kaggle_file.csv')` to use a real dataset. Columns needed: `text`, `category`.

In [6]:
from google.colab import files
# Checking files and loading the correct one
import os

file_name = 'all_tickets_processed_improved_v3.csv'
if not os.path.exists(file_name):
    # Fallback if a duplicate was uploaded
    file_name = 'all_tickets_processed_improved_v3 (1).csv'

df = pd.read_csv(file_name)
print(f"Loaded {file_name}")
display(df.shape)

Loaded all_tickets_processed_improved_v3.csv


(47837, 2)

In [7]:
df.head()

,Document,Topic_group
0,connection with icon icon dear please setup icon per ico...,Hardware
1,work experience user work experience user hi work experi...,Access
2,requesting for meeting requesting meeting hi please help...,Hardware
3,reset passwords for external accounts re expire days hi ...,Access
4,mail verification warning hi has got attached please add...,Miscellaneous


## 2. Clean data

In [8]:
df = df.dropna(subset=['Document', 'Topic_group'])
df['text'] = df['Document'].str.strip().str.lower()
df['category'] = df['Topic_group']
df = df[df['text'].str.len() > 0]
df.shape

(47837, 4)

## 3. EDA — class balance

In [9]:
df['category'].value_counts()

,count
category,
Hardware,13617
HR Support,10915
Access,7125
Miscellaneous,7060
Storage,2777
Purchase,2464
Internal Project,2119
Administrative rights,1760


In [10]:
df['text'].str.split().apply(len).describe()

,text
count,47837.000000
mean,43.597341
std,56.736800
min,2.000000
25%,17.000000
50%,26.000000
75%,46.000000
max,981.000000


## 4. Train / validation / test split (70/15/15)

In [18]:
X_train, X_temp, y_train, y_temp = train_test_split(df['text'], df['category'], test_size=0.3, random_state=42, stratify=df['category'])
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
print(f"Train size: {len(X_train)}, Val size: {len(X_val)}, Test size: {len(X_test)}")

Train size: 33485, Val size: 7176, Test size: 7176


## 5. Baseline model — TF-IDF + Logistic Regression

In [12]:
vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1,2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

baseline = LogisticRegression(max_iter=1000)
baseline.fit(X_train_vec, y_train)

pred_baseline = baseline.predict(X_test_vec)
acc_baseline = accuracy_score(y_test, pred_baseline)
p, r, f1, _ = precision_recall_fscore_support(y_test, pred_baseline, average='weighted')
print(f"Accuracy: {acc_baseline:.3f}  Precision: {p:.3f}  Recall: {r:.3f}  F1: {f1:.3f}")

Accuracy: 0.849  Precision: 0.855  Recall: 0.849  F1: 0.849


In [13]:
print(confusion_matrix(y_test, pred_baseline))
print(classification_report(y_test, pred_baseline))

[[ 895    2   44   83    4   36    0    5]
 [   9  169   11   70    1    4    0    0]
 [  21    1 1428  135    4   44    0    5]
 [  32   20   85 1832    4   61    6    2]
 [   5    1   27   32  241   12    0    0]
 [  13    1   58  107    4  872    0    4]
 [   2    4    5   34    0    4  320    1]
 [   4    2   21   46    0    9    0  334]]
                       precision    recall  f1-score   support

               Access       0.91      0.84      0.87      1069
Administrative rights       0.84      0.64      0.73       264
           HR Support       0.85      0.87      0.86      1638
             Hardware       0.78      0.90      0.84      2042
     Internal Project       0.93      0.76      0.84       318
        Miscellaneous       0.84      0.82      0.83      1059
             Purchase       0.98      0.86      0.92       370
              Storage       0.95      0.80      0.87       416

             accuracy                           0.85      7176
            macro avg  

## 6. Improved model — Linear SVM

In [14]:
svm = LinearSVC()
svm.fit(X_train_vec, y_train)

pred_svm = svm.predict(X_test_vec)
acc_svm = accuracy_score(y_test, pred_svm)
p2, r2, f1_2, _ = precision_recall_fscore_support(y_test, pred_svm, average='weighted')
print(f"Accuracy: {acc_svm:.3f}  Precision: {p2:.3f}  Recall: {r2:.3f}  F1: {f1_2:.3f}")

Accuracy: 0.855  Precision: 0.857  Recall: 0.855  F1: 0.855


## 7. Before vs after comparison

In [15]:
comparison = pd.DataFrame({
    'Model': ['Logistic Regression (baseline)', 'Linear SVM (improved)'],
    'Accuracy': [acc_baseline, acc_svm],
    'F1': [f1, f1_2]
})
comparison

,Model,Accuracy,F1
0,Logistic Regression (baseline),0.848802,0.848882
1,Linear SVM (improved),0.854654,0.854859


## 8. Error analysis

In [16]:
errors = pd.DataFrame({'text': X_test, 'true': y_test, 'pred': pred_svm})
errors = errors[errors['true'] != errors['pred']]
errors.head(5)

,text,true,pred
26218,add afield to the notification tuesday november pm re re...,Access,Miscellaneous
18584,oracle fusion down time notification thursday march pm d...,HR Support,Hardware
4879,recent oracle security bug announcement recent announcem...,Hardware,HR Support
39487,re request extra disks external hardware has been assign...,Miscellaneous,Hardware
43041,issue updating career development discussion sent thursd...,HR Support,Miscellaneous


## 9. Save final model + vectorizer

In [17]:
joblib.dump(svm, 'ticket_model.joblib')
joblib.dump(vectorizer, 'ticket_vectorizer.joblib')
files.download('ticket_model.joblib')
files.download('ticket_vectorizer.joblib')
print('Saved and downloaded.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved and downloaded.
